In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer
#ds = load_dataset("codeparrot/apps", split="train")

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")


/local/scratch/droytbu/.hackenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [38]:
import json
mbpp_articles = {}
mbpp_code = {}
with open("mbpp.jsonl","r") as f:
    for item in f.readlines():
        item = json.loads(item)
        mbpp_articles[item['task_id']] = item['text'] + "\n\n Examples: \n" + "\n".join(item['test_list'])
        mbpp_code[item['task_id']] = item['code']
json.dump(mbpp_articles, open("../articles/mbpp_train_articles.json","w"))
json.dump(mbpp_code, open("../summaries/mbpp/mbpp_train_human_responses.json","w"))


In [65]:
from datasets import load_dataset

In [67]:
arena = load_dataset("lmarena-ai/arena-human-preference-100k",split="train")

Generating train split: 100%|██████████| 106134/106134 [00:02<00:00, 46683.98 examples/s]


In [106]:
arena.features

{'question_id': Value(dtype='string', id=None),
 'model_a': Value(dtype='string', id=None),
 'model_b': Value(dtype='string', id=None),
 'winner': Value(dtype='string', id=None),
 'conversation_a': [{'content': Value(dtype='string', id=None),
   'num_tokens': Value(dtype='int64', id=None),
   'role': Value(dtype='string', id=None)}],
 'conversation_b': [{'content': Value(dtype='string', id=None),
   'num_tokens': Value(dtype='int64', id=None),
   'role': Value(dtype='string', id=None)}],
 'turn': Value(dtype='int64', id=None),
 'anony': Value(dtype='bool', id=None),
 'language': Value(dtype='string', id=None),
 'tstamp': Value(dtype='float64', id=None),
 'conv_metadata': {'bold_count_a': {'**': Value(dtype='int64', id=None),
   '__': Value(dtype='int64', id=None)},
  'bold_count_b': {'**': Value(dtype='int64', id=None),
   '__': Value(dtype='int64', id=None)},
  'context_a_tokens': Value(dtype='int64', id=None),
  'context_b_tokens': Value(dtype='int64', id=None),
  'header_count_a': {

In [69]:
data = {}
for model in ["llama-3.1-8b-instruct", "llama-3.1-70b-instruct", "llama-3.1-405b-instruct"]:
    data[model] = arena.filter(lambda a: a['model_a'] == model or a['model_b'] == model)

Filter: 100%|██████████| 106134/106134 [00:15<00:00, 6744.84 examples/s]


In [108]:
ia = 0
j = 0
for model in ["llama-3.1-8b-instruct", "llama-3.1-70b-instruct", "llama-3.1-405b-instruct"]:
    model_data = []
    for entry in data[model]:
        add = {}
        assert entry['conversation_a'][0]['content'] == entry['conversation_b'][0]['content']
        if not len(entry['conversation_a']) == len(entry['conversation_b']) == 2:
            ia += 1
            #print("NEW\tNEW\tNEW")
            for i in range(0, len(entry['conversation_a']), 2):
                a, b = entry['conversation_a'][i], entry['conversation_b'][i]
                #print(a['content'], b['content'])
                assert a['content'] == b['content']
                #print("\n\n")
            continue

        own = 'a' if entry['model_a'] == model else 'b'
        other = 'b' if own == 'a' else 'a'

        add['id'] = entry['question_id']
        add['self'] = entry[f"model_{own}"]
        add['other'] = entry[f'model_{other}']
        add['prompt'] = entry[f'conversation_{own}'][0]['content']
        add['self_response'] = entry[f'conversation_{own}'][-1]['content']
        add['other_response'] = entry[f'conversation_{other}'][-1]['content']
        add['won'] = 1 if entry['winner'] == f"model_{own}" else 0
        add['language'] = entry['language']

        #print(add)
        model_data.append(add)
    pd.DataFrame(model_data).to_csv(f"../arena_data/chat_arena/{model}_preference_data.csv")


In [54]:
from datasets import load_dataset, get_dataset_config_names
config_options = get_dataset_config_names("cais/mmlu")
config_options = [op for op in config_options if op not in ["all", "all_train", "auxiliary_train"]]
for option in config_options:
    mmlu_opt_articles = {}
    mmlu_opt_responses = {}
    mmlu_opt = load_dataset("cais/mmlu", option)
    for split in mmlu_opt.keys():
        for i, entry in enumerate(mmlu_opt[split]):
            id = split + "_" + str(i)
            mmlu_opt_articles[id] = entry['question'] + "\n\nOptions: " + "\n".join(entry['choices'])
            mmlu_opt_responses[id] = entry['answer']
    json.dump(mmlu_opt_articles, open(f"../articles/mmlu_{option}_train_articles.json","w"))
    if not os.path.isdir(f"../summaries/mmlu_{option}"):
        os.mkdir(f"../summaries/mmlu_{option}")
    json.dump(mmlu_opt_responses, open(f"../summaries/mmlu_{option}/mmlu_{option}_train__human_responses.json","w"))


Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 1821.71 examples/s]
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml
Retrying in 4s [Retry 3/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml
Retrying in 8s [Retry 4/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/cais/mmlu/resolve/c30699e8356da336a370243923dbaf21066bb9fe/.huggingface.yaml
Retrying in 8s [Retry 5/5].
HTTP Error 429 thr

ValueError: Couldn't find cache for cais/mmlu for config 'management'
Available configs in the cache: ['abstract_algebra', 'all', 'anatomy', 'astronomy', 'auxiliary_train', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning']

In [1]:
import os
os.environ['HF_HOME'] = "/local/scratch/droytbu/self_recognition/bigpatent"

In [10]:
openpatent = load_dataset("NortheasternUniversity/big_patent",split="train")


OSError: [Errno 122] Disk quota exceeded

In [11]:
!export HF_HOME="/local/scratch/droytbu/self_recognition"

In [15]:
ds['difficulty']

['interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'interview',
 'inte

In [42]:
dataset = {}
answers = {}
THRESHOLD=300
import json
from tqdm import tqdm
for input in tqdm(ds): 
    if input['difficulty'] != 'interview':
        print(input['difficulty'])
        continue
    #formatting the "articles" (inputs to models)
    question = ''
    question += "Problem: \n" + input['question']
    question += "\n\n Examples: \n"
    try:
        i_o = json.loads(input['input_output'])
        if len(i_o['inputs']) > 0 and len(i_o['outputs']) > 0:
            #inputs, outputs = [i.strip("][ ") for i in i_o['inputs'].split()], [i.strip("][ ") for i in i_o['inputs'].split()] 
            for i,o in zip(i_o['inputs'], i_o['outputs']):
                question += "Input: \n" + str(i) + "\nOutput: \n " + str(o) + "\n"
    except json.JSONDecodeError:
        continue


    #formatting the "summaries" (outputs from humans)
    tok_len = len(tokenizer.tokenize(eval(input['solutions'])[0]))
    if tok_len < THRESHOLD:
        dataset[input['problem_id']] = question
        answers[input['problem_id']] = eval(input['solutions'])[0]
    
with open("apps_train_code.json", "w") as f:
    json.dump(dataset, f)
with open("../summaries/apps/apps_train_human_responses_merged.json", "w") as f:
    json.dump(dataset, f)

 66%|██████▌   | 3308/5000 [00:01<00:00, 4354.63it/s]

competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
competition
comp

100%|██████████| 5000/5000 [00:02<00:00, 2370.97it/s]

introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory
introductory

In [ ]:
import json
from datasets import load_dataset
from transformers import *
medmcqa = load_dataset("openlifescienceai/medmcqa")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

In [ ]:
import json
from datasets import load_dataset
from transformers import *
from tqdm import tqdm

medmcqa = load_dataset("openlifescienceai/medmcqa")
medmcqa_json = {}; medmcqa_answers = {}
ref = ['a','b','c','d']
for example in tqdm(medmcqa['train']):
    correct_answer = 'op' + ref[example['cop']]
    medmcqa_json[example['id']] = \
    f"""{example['question']}
        Correct Answer: {example[correct_answer]}

        Can you explain why this is the case?
    """
    print(medmcqa_json[example['id']])
    medmcqa_answers[example['id']] = example['exp']


  1%|          | 1458/182822 [00:00<00:24, 7308.21it/s]

Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma
        Correct Answer: Atrophy

        Can you explain why this is the case?
    
Which vitamin is supplied from only animal source:
        Correct Answer: Vitamin B12

        Can you explain why this is the case?
    
All of the following are surgical options for morbid obesity except -
        Correct Answer: Roux en Y Duodenal By pass

        Can you explain why this is the case?
    
Following endaerectomy on the right common carotid, a patient is found to be blind in the right eye. It is appears that a small thrombus embolized during surgery and lodged in the aery supplying the optic nerve. Which aery would be blocked?
        Correct Answer: Central aery of the retina

        Can you explain why this is the case?
    
Growth hormone has its effect on growth through?
        Correct Answer: IG1-1

        Can you explain why this is the case?
    
Scrub typh

  2%|▏         | 2940/182822 [00:00<00:24, 7258.33it/s]

Nerve compressed by aneurysm of posterior communicating aery is
        Correct Answer: Occulomotor nerve

        Can you explain why this is the case?
    
Curschmann spirals are found in
        Correct Answer: Asthma

        Can you explain why this is the case?
    
All are TRUE about the relation of inguinal canal, EXCEPT:
        Correct Answer: Interfoveolar ligament forms lateral two third of anterior wall

        Can you explain why this is the case?
    
In emergency tracheostomy all of the following structures are damaged, EXCEPT:
        Correct Answer: Inferior thyroid aery

        Can you explain why this is the case?
    
Platelets transfusion must be completed in how many hours after entering the bag
        Correct Answer: 4 hour

        Can you explain why this is the case?
    
Rickets in infant present as all except -
        Correct Answer: Bow legs

        Can you explain why this is the case?
    
The maxillary nerve arises from the trigeminal ganglion in t

  3%|▎         | 4676/182822 [00:00<00:21, 8111.03it/s]

...

 14%|█▍        | 25711/182822 [00:02<00:17, 8885.81it/s]

Test used to differentiate maternal from fetal blood
        Correct Answer: Apt test

        Can you explain why this is the case?
    
Patient should be kept nil orally for:
        Correct Answer: 6 hours

        Can you explain why this is the case?
    
A chronic alcoholic develops palpitations suddenly after alcohol binge. Which of the following arrythmia is most commonly associated with alcohol binge in the alcoholics?
        Correct Answer: Atrial fibrillation

        Can you explain why this is the case?
    
What is the major advantage of randomised sample in a clinical trial?
        Correct Answer: Reduce selection bias in allocation of treatment

        Can you explain why this is the case?
    
A 34-year-old man complains of hearing loss. He has had multiple bouts of ear infections over the last 20 years and was recently diagnosed with chronic suppurative otitis media. Which of the following is the most likely complication of this condition in this patient?
        C

 14%|█▍        | 26488/182822 [00:03<00:18, 8664.35it/s]


Insulin of choice for the treatment of diabetes mellitus is:
        Correct Answer: Regular Insulin

        Can you explain why this is the case?
    
The DMF index when used for primary teeth, uses
        Correct Answer: Lower case letters

        Can you explain why this is the case?
    
Death of a person due to compressing of neck by another person is -
        Correct Answer: Throttling

        Can you explain why this is the case?
    
Negri bodies are characteristic of: September 2008, March 2013
        Correct Answer: Rabies

        Can you explain why this is the case?
    
Which anaesthetic agent does not affect blood flow to liver?
        Correct Answer: Isoflurane

        Can you explain why this is the case?
    
Vaccine for yellow fever -
        Correct Answer: 17 D

        Can you explain why this is the case?
    
Which of the following is true about wandering fibroid?
        Correct Answer: Attached to surrounding viscera

        Can you explain why this i

KeyboardInterrupt: 

In [8]:
os.makedirs("../verifiable_articles/medmcqa")

In [68]:
with open("../articles/medmcqa_train_articles.json","w") as f:
    json.dump(medmcqa_json, f)
with open("../summaries/medmcqa/medmcqa_train_human_responses_merged.json","w") as f:
    json.dump(medmcqa_answers, f)

In [1]:
import os
import os.path as osp
import pandas as pd

og_folder = "/local/scratch/droytbu/self_recognition/experiments/cnn/cnn_n300_m6_20250701-1715"
for model in os.listdir(og_folder):
    if osp.isdir(osp.join(og_folder, model)) and model != "heatmaps":
        print(model)
        model_df = pd.read_csv(osp.join(og_folder, model, f"{model}_comparison_results_self_prefer_rate_simple.csv"))


deepseek-v3-0324
gpt35
llama-4-scout-17b-16e-instruct
llama3.3-70b-instruct-fp8
llama3.1-8b-instruct
hermes3-405b
